<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/CLIP_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/torch_clip.pth'
save_path_vit = '/content/drive/MyDrive/torch_clip_vit.pth'
save_path_vit_transformer = '/content/drive/MyDrive/torch_clip_vit_transformer.pth'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
max_length = 30
lstm_num_layers = 2
embed_dim = 1000 # TODO, add model output embedding and project embedding
transformer_embed_dim = 512
hidden_dim = 256
batch_size = 32
num_epochs = 5
num_heads = 8
transformer_layers = 3
transformer_dropout = 0.1
learning_rate = 1e-4
W = 224
H = 224
temperature = 0.07
Image_Net_Mean = [0.485, 0.456, 0.406]
Image_Net_Std = [0.229, 0.224, 0.225]
image_model_name = "vit"
text_model_name = "transformer"

In [ ]:
# Download the train images (this is a large file, ~18GB)
!wget http://images.cocodataset.org/zips/train2017.zip

# Download the annotations (contains the captions JSON file)
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# Unzip the images
!unzip -q train2017.zip

# Unzip the annotations
!unzip -q annotations_trainval2017.zip

--2025-04-13 07:44:51--  http://images.cocodataset.org/zips/train2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 3.5.27.43, 3.5.27.255, 16.15.193.255, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|3.5.27.43|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19336861798 (18G) [application/zip]
Saving to: ‘train2017.zip’

train2017.zip       100%[===================>]  18.01G   104MB/s    in 3m 29s  

2025-04-13 07:48:20 (88.4 MB/s) - ‘train2017.zip’ saved [19336861798/19336861798]

--2025-04-13 07:48:20--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.179.3, 3.5.0.211, 3.5.27.36, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.179.3|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘annotations_trainval2017.zip’

annotations_trainva 100%[====

In [ ]:
!pip install pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.7/458.7 kB 6.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms
from torchvision.datasets import CocoCaptions
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import os

########################################
# Vocabulary and Tokenization Utilities
########################################

def build_vocab(captions, threshold=1):
    """Build a simple vocabulary from a list of captions."""
    counter = Counter()
    for caption in captions:
        tokens = caption.lower().split()
        counter.update(tokens)
    # Reserve indices for special tokens
    vocab = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
    idx = 4
    for word, count in counter.items():
        if count >= threshold:
            vocab[word] = idx
            idx += 1
    return vocab

def tokenize_caption(caption, vocab, max_length):
    """Convert a caption string into a list of token IDs."""
    tokens = caption.lower().split()
    tokens = ["<start>"] + tokens + ["<end>"]
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    # Pad or truncate
    if len(token_ids) < max_length:
        token_ids += [vocab["<pad>"]] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]
    return torch.tensor(token_ids)

########################################
# Dataset Definition for COCO Captions
########################################

class COCOClipDataset(Dataset):
    def __init__(self, root, annFile, vocab, transform=None, max_length=30, train=True):
        """
        Wraps the CocoCaptions dataset:
          - root: folder with images
          - annFile: path to the JSON annotations file
          - vocab: vocabulary dictionary for tokenizing captions
          - transform: image transforms
          - max_length: maximum sequence length for captions
        """
        self.coco = CocoCaptions(root=root, annFile=annFile, transform=transform)
        self.vocab = vocab
        self.max_length = max_length
        self.size = len(self.coco)
        self.train = train
        self.train_size = int(0.8 * self.size)
        self.val_size = self.size - self.train_size

    def __len__(self):
        if self.train:
            return self.train_size
        else:
            return self.val_size

    def __getitem__(self, idx):
        index = idx if self.train else idx + self.train_size
        img, captions = self.coco[index]
        # For simplicity, use the first caption
        caption = captions[0]
        token_ids = tokenize_caption(caption, self.vocab, self.max_length)
        return img, token_ids

########################################
# Model Definitions
########################################

# Text Encoder using an Embedding layer and LSTM
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super(TextEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x):
        # x: (batch, seq_length)
        x = self.embedding(x)  # (batch, seq_length, embed_dim)
        out, (h_n, _) = self.lstm(x)
        h = h_n[-1]  # take the last layer's hidden state: (batch, hidden_dim)
        x = self.fc(h)  # (batch, embed_dim)
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape becomes (1, max_len, embed_dim)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_length, embed_dim)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class TransformerTextTEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, output_embed_dim, num_heads, num_layers, dropout=0.1):
        super(TransformerTextTEncoder, self).__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim, dropout)

        # Define a transformer encoder layer and stack with TransformerEncoder.
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        # Fully connected layer mapping the pooled transformer output to embed_dim
        self.fc = nn.Linear(embed_dim, output_embed_dim)

    def forward(self, x):
        x = self.embedding(x) * math.sqrt(self.embed_dim)  # (batch, seq_length, embed_dim)
        x = self.pos_encoder(x)
        x = x.transpose(0, 1)
        transformer_out = self.transformer_encoder(x)  # (seq_length, batch, embed_dim)
        pooled_out = transformer_out.mean(dim=0)  # (batch, embed_dim)
        output = self.fc(pooled_out)  # (batch, embed_dim)
        return output

# Image Encoder: Using ResNet-18 with a modified final layer
def get_image_encoder(embed_dim, image_model_name="resnet"):
    model = None
    if image_model_name == "resnet":
      model = models.resnet18(pretrained=True)

      for param in model.parameters():
          param.requires_grad = False

      num_features = model.fc.in_features
      #model.fc = nn.Linear(num_features, embed_dim)
      model.fc = nn.Identity()
    elif image_model_name == "vit":
      model = models.vit_b_16(pretrained=True)
      for param in model.parameters():
          param.requires_grad = False
      model.head = nn.Identity()
    return model

# CLIP-like Model that jointly encodes images and text into the same space
class CLIPModel(nn.Module):
    def __init__(self, image_encoder, text_encoder, embed_dim):
        super(CLIPModel, self).__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        self.embed_dim = embed_dim

    def forward(self, images, texts):
        # Encode images and text
        image_features = self.image_encoder(images)   # (batch, embed_dim)
        image_features = torch.flatten(image_features, 1)
        text_features = self.text_encoder(texts)        # (batch, embed_dim)
        # Normalize the embeddings to unit length
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return image_features, text_features

########################################
# Contrastive Loss Function
########################################

def clip_loss(image_features, text_features):
    """
    Computes the CLIP contrastive loss.
    For a batch of size B, we compute the similarity matrix between all image-text pairs.
    """
    image_features = F.normalize(image_features, p=2, dim=1)
    text_features = F.normalize(text_features, p=2, dim=1)
    logits = torch.matmul(image_features, text_features.t()) / temperature
    logits = F.log_softmax(logits, dim=-1)

    logits_t = logits.t()
    logits_t = F.log_softmax(logits_t, dim=-1)

    image_logits = torch.matmul(image_features, image_features.t()) / temperature
    image_logits = F.softmax(image_logits, dim=-1)

    loss_i2t = F.kl_div(logits, image_logits, reduction='batchmean')
    loss_t2i = F.kl_div(logits_t, image_logits, reduction='batchmean')
    loss = (loss_i2t + loss_t2i) / 2
    return loss

########################################
# Training Setup
########################################

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define image transforms
transform = transforms.Compose([
    transforms.Resize((W, H)),
    transforms.ToTensor(),
    transforms.Normalize(mean=Image_Net_Mean,
                         std=Image_Net_Std)
])

# Update these paths to point to your local COCO dataset
coco_root = "train2017"  # e.g. "./data/train2017"
annFile = "annotations/captions_train2017.json"  # e.g. "./data/annotations/captions_train2017.json"

# Build vocabulary from a subset of captions (for demonstration purposes)
print("Building vocabulary...")
coco_temp = CocoCaptions(root=coco_root, annFile=annFile, transform=transform)
all_captions = []
num_samples_for_vocab = 5000  # Use a subset to keep it fast
for i in range(min(len(coco_temp), num_samples_for_vocab)):
    _, captions = coco_temp[i]
    all_captions.extend(captions)
vocab = build_vocab(all_captions, threshold=1)
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

# Create dataset and DataLoader
dataset = COCOClipDataset(root=coco_root, annFile=annFile, vocab=vocab, transform=transform, max_length=max_length)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

test_dataset = COCOClipDataset(root=coco_root, annFile=annFile, vocab=vocab, transform=transform, max_length=max_length, train=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# Initialize encoders and the CLIP model
image_encoder = get_image_encoder(embed_dim, image_model_name=image_model_name).to(device)
if text_model_name == "transformer":
  text_encoder = TransformerTextTEncoder(vocab_size, transformer_embed_dim, embed_dim, num_heads, transformer_layers, transformer_dropout).to(device)
else:
  text_encoder = TextEncoder(vocab_size, embed_dim, hidden_dim, num_layers=lstm_num_layers).to(device)
model = CLIPModel(image_encoder, text_encoder, embed_dim).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Building vocabulary...
loading annotations into memory...
Done (t=0.84s)
creating index...
index created!
Vocabulary size: 10074
loading annotations into memory...
Done (t=0.84s)
creating index...
index created!
loading annotations into memory...
Done (t=0.99s)
creating index...
index created!


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [ ]:
########################################
# Training Loop
########################################

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for i, (images, captions) in enumerate(dataloader):
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()
        image_features, text_features = model(images, captions)
        loss = clip_loss(image_features, text_features)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(dataloader)}], Loss: {loss.item()}")
            torch.save(model, save_path_vit_transformer)

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {avg_loss}")

torch.save(model, save_path_vit_transformer)
print("Training completed.")

Epoch [1/5], Step [50/2958], Loss: 2.2763867378234863
Epoch [1/5], Step [100/2958], Loss: 2.2818994522094727
Epoch [1/5], Step [150/2958], Loss: 1.6713480949401855
Epoch [1/5], Step [200/2958], Loss: 1.7111775875091553
Epoch [1/5], Step [250/2958], Loss: 1.6555473804473877
Epoch [1/5], Step [300/2958], Loss: 1.3672335147857666
Epoch [1/5], Step [350/2958], Loss: 1.6367069482803345
Epoch [1/5], Step [400/2958], Loss: 1.4599242210388184
Epoch [1/5], Step [450/2958], Loss: 1.3303065299987793
Epoch [1/5], Step [500/2958], Loss: 1.4240761995315552
Epoch [1/5], Step [550/2958], Loss: 1.1292659044265747
Epoch [1/5], Step [600/2958], Loss: 1.0822250843048096
Epoch [1/5], Step [650/2958], Loss: 1.2563509941101074
Epoch [1/5], Step [700/2958], Loss: 0.9490867853164673
Epoch [1/5], Step [750/2958], Loss: 1.064833641052246
Epoch [1/5], Step [800/2958], Loss: 0.9439381957054138
Epoch [1/5], Step [850/2958], Loss: 0.949816882610321
Epoch [1/5], Step [900/2958], Loss: 1.0519323348999023
Epoch [1/5], 

In [ ]:
import torch

def evaluate_top_k(model, dataloader, device, k=100):
    model.eval()
    image_embeddings = []
    text_embeddings = []

    # Compute embeddings for the entire test set
    with torch.no_grad():
        for images, captions in dataloader:
            images = images.to(device)
            captions = captions.to(device)
            img_emb, txt_emb = model(images, captions)
            img_emb = F.normalize(img_emb, p=2, dim=-1)
            txt_emb = F.normalize(txt_emb, p=2, dim=-1)
            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    # Compute pairwise similarity between each image and text embedding
    # Here we use a simple dot product; embeddings are assumed to be normalized.
    similarity_matrix = torch.matmul(image_embeddings, text_embeddings.t())

    # For each image, get the indices of the top-k highest scoring text embeddings
    topk_values, topk_indices = similarity_matrix.topk(k, dim=1)

    # Example: If each image has a corresponding caption at the same index,
    # we check if the correct caption index is in the top k indices.
    correct = 0
    total = similarity_matrix.size(0)
    for i in range(total):
        if i in topk_indices[i]:
            correct += 1

    recall_at_k = correct / total
    print(f"Recall@{k}: {recall_at_k:.4f}")

    return topk_indices

# Example usage:
# Assuming you have a test dataloader similar to your training one
# and 'model' is your trained CLIPModel.
top_100_indices = evaluate_top_k(model, test_dataloader, device, k=100)


Recall@100: 0.5319
